# CENG 467 — Question 3: Text Summarization

**Author:** Yusuf Furkan Aktay  
**Course:** CENG 467 — Natural Language Understanding and Generation  
**Instructor:** Prof. Dr. Aytuğ Onan

## Objective
Compare extractive (LexRank) and abstractive (BART-CNN) summarization approaches on the CNN/DailyMail dataset, evaluated with ROUGE, BLEU, METEOR, and BERTScore.

## Setup
- **Runtime:** Google Colab — GPU (T4)
- **Random Seed:** 42
- **Dataset:** CNN/DailyMail (3.0.0) — 100-article subset of test split
- **Models:** LexRank (extractive), `facebook/bart-large-cnn` (abstractive, pre-trained, no fine-tuning)

Run `Runtime → Run all` after selecting GPU.

## 1. Environment Setup

In [ ]:
!pip install -q --upgrade transformers accelerate datasets
!pip install -q rouge-score sacrebleu nltk bert-score sumy
# IMPORTANT: After this cell, do Runtime → Restart session, then Run all again.

In [ ]:
import os, random, json, time, warnings
import numpy as np
import pandas as pd
import torch
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

import nltk
nltk.download('punkt', quiet=True); nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True); nltk.download('omw-1.4', quiet=True)

os.makedirs('results', exist_ok=True)

## 2. Dataset: CNN/DailyMail

CNN/DailyMail (v3.0.0) is a benchmark for news summarization, containing news articles paired with multi-sentence reference summaries ("highlights"). The full test split contains 11,490 articles. Following the assignment's allowance for subsets, we use a **100-article test subset** for tractable evaluation under Colab Free constraints. The subset is sampled with a fixed seed for reproducibility.

In [ ]:
from datasets import load_dataset
ds = load_dataset('abisee/cnn_dailymail', '3.0.0', split='test')
ds = ds.shuffle(seed=SEED).select(range(100))
print(f'Test subset size: {len(ds)}')
print(f'Avg article length (chars): {np.mean([len(x["article"]) for x in ds]):.0f}')
print(f'Avg summary length (chars): {np.mean([len(x["highlights"]) for x in ds]):.0f}')
print('\n--- Sample ---')
print('ARTICLE :', ds[0]['article'][:300], '...')
print('\nREFERENCE:', ds[0]['highlights'])

## 3. Model 1 — LexRank (Extractive)

LexRank is a graph-based unsupervised extractive method. It builds a graph where nodes are sentences and edges are weighted by cosine similarity (TF-IDF), then runs PageRank to identify the most central sentences. We extract the top-3 sentences per article (matching CNN/DM's typical 3-sentence reference summary).

In [ ]:
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer

lex_summarizer = LexRankSummarizer()
NUM_SENT = 3  # CNN/DM references are ~3-4 sentences

def lexrank_summarize(text, n=NUM_SENT):
    parser = PlaintextParser.from_string(text, Tokenizer('english'))
    summary_sentences = lex_summarizer(parser.document, n)
    return ' '.join(str(s) for s in summary_sentences)

t0 = time.time()
lexrank_preds = [lexrank_summarize(x['article']) for x in ds]
lex_time = time.time() - t0
print(f'LexRank time: {lex_time:.1f}s for {len(ds)} articles')
print('\n--- Sample LexRank summary ---')
print(lexrank_preds[0])

## 4. Model 2 — BART-CNN (Abstractive)

We use `facebook/bart-large-cnn`, a BART model already fine-tuned on CNN/DailyMail. We perform inference only — no further fine-tuning — which is standard practice when comparing methods under compute constraints.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm

BART_NAME = 'facebook/bart-large-cnn'
bart_tok = AutoTokenizer.from_pretrained(BART_NAME)
bart_model = AutoModelForSeq2SeqLM.from_pretrained(BART_NAME).to(DEVICE)
bart_model.eval()

def bart_summarize(text, max_len=130, min_len=30):
    inputs = bart_tok([text], max_length=1024, truncation=True, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = bart_model.generate(**inputs, max_length=max_len, min_length=min_len,
                                   num_beams=4, length_penalty=2.0, early_stopping=True)
    return bart_tok.decode(out[0], skip_special_tokens=True)

t0 = time.time()
bart_preds = []
for x in tqdm(ds, desc='BART-CNN inference'):
    bart_preds.append(bart_summarize(x['article']))
bart_time = time.time() - t0
print(f'\nBART time: {bart_time:.1f}s for {len(ds)} articles')
print('\n--- Sample BART summary ---')
print(bart_preds[0])

## 5. Evaluation: ROUGE, BLEU, METEOR, BERTScore

In [ ]:
from rouge_score import rouge_scorer
import sacrebleu
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
from bert_score import score as bert_score_fn

refs = [x['highlights'] for x in ds]

def evaluate(preds, refs, label):
    # ROUGE
    scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
    r1, r2, rl = [], [], []
    for p, r in zip(preds, refs):
        s = scorer.score(r, p)
        r1.append(s['rouge1'].fmeasure); r2.append(s['rouge2'].fmeasure); rl.append(s['rougeL'].fmeasure)

    # BLEU (corpus-level)
    bleu = sacrebleu.corpus_bleu(preds, [refs]).score / 100.0

    # METEOR (sentence-level mean)
    meteor_scores = []
    for p, r in zip(preds, refs):
        meteor_scores.append(meteor_score([word_tokenize(r)], word_tokenize(p)))

    # BERTScore (F1)
    P, R, F1 = bert_score_fn(preds, refs, lang='en', verbose=False, device=DEVICE.type)
    bertscore_f1 = F1.mean().item()

    out = {
        'Model': label,
        'ROUGE-1': np.mean(r1),
        'ROUGE-2': np.mean(r2),
        'ROUGE-L': np.mean(rl),
        'BLEU':    bleu,
        'METEOR':  np.mean(meteor_scores),
        'BERTScore-F1': bertscore_f1,
    }
    return out

lex_metrics  = evaluate(lexrank_preds, refs, 'LexRank (extractive)')
bart_metrics = evaluate(bart_preds,    refs, 'BART-CNN (abstractive)')

summary = pd.DataFrame([lex_metrics, bart_metrics])
summary['Time (s)'] = [lex_time, bart_time]
print(summary.to_string(index=False))
summary.to_csv('results/q3_summary.csv', index=False)

## 6. Qualitative Analysis (3 Examples)

In [ ]:
examples = []
for i in [0, 1, 2]:
    ex = {
        'article_excerpt': ds[i]['article'][:500] + '...',
        'reference': refs[i],
        'lexrank':  lexrank_preds[i],
        'bart':     bart_preds[i],
    }
    examples.append(ex)
    print(f'\n========== EXAMPLE {i+1} ==========')
    print(f"\n[ARTICLE EXCERPT]\n{ex['article_excerpt']}")
    print(f"\n[REFERENCE]\n{ex['reference']}")
    print(f"\n[LEXRANK (extractive)]\n{ex['lexrank']}")
    print(f"\n[BART-CNN (abstractive)]\n{ex['bart']}")

with open('results/q3_examples.json','w') as f:
    json.dump(examples, f, indent=2)
with open('results/q3_full.json','w') as f:
    json.dump({'lexrank': lex_metrics, 'bart': bart_metrics,
               'lex_time': lex_time, 'bart_time': bart_time}, f, indent=2)
print('\nResults saved to results/')

## 7. Discussion (in Report)

See `reports/CENG467_Midterm_Report.pdf` for the comparison of extractive vs. abstractive summarization in terms of:
- Fluency (BART produces more natural prose; LexRank inherits sentence breaks from source)
- Factual consistency (extractive guarantees factual fidelity; abstractive can hallucinate)
- Information coverage (BART distills cross-sentence info; LexRank limited to existing sentences)
- Computational cost (LexRank is ~100× faster but less fluent)